---
title: "Chapter 33: Predicting household spend"
---

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sambaiga/ds-mlops-path/blob/main/tutorials/05-classical-ml-projects/33-spend-regression.ipynb) [![Download Notebook](https://img.shields.io/badge/Download-Notebook-blue.svg?logo=jupyter&logoColor=white)](https://raw.githubusercontent.com/sambaiga/ds-mlops-path/main/tutorials/05-classical-ml-projects/33-spend-regression.ipynb)

In November 2019, a single-person household in Alabama, reporting an income between $25,000 and $49,999, bought a 65-inch Samsung 4K television, a PlayStation 4 Pro, a Microsoft Surface tablet, an Apple Watch, a GoPro camera bundle, a trampoline, and a kayak. Fifty-eight items, spread across four days. The total came to $11,709.16.

For the three months before that, the same household had spent an average of $49.47 a month. Across the roughly four years this account appears in the data, its lifetime total is $16,237.77. One month accounts for 72 percent of everything it has ever bought on the platform.

Nothing in this household's survey answers, age bracket, income, education, state, explains it. There's no flag for it and no obvious trigger. It's one row among 1.85 million, sitting in the data whether or not anyone thinks to look for it.

Chapters 28 through 32 gave you the machinery for exactly this kind of question: how to turn a vague business problem into [a five-component specification](../04-ml-intro/30-ml-workflow.qmd#translating-a-business-problem-into-a-machine-learning-problem), how [the nine-stage workflow](../04-ml-intro/30-ml-workflow.qmd#the-end-to-end-machine-learning-workflow) moves data through a model and into production, how scikit-learn's [Transformer and Predictor interfaces](../04-ml-intro/31-sklearn-core.ipynb) work, how [a Pipeline strings them together](../04-ml-intro/32-sklearn-pipeline.ipynb) without leaking information across a split. You used all of it on hotel booking data that was already clean enough to teach with.

Here, you use it on a dataset nobody cleaned for teaching: five years of real Amazon purchase histories from 5,027 U.S. households, linked to a demographic and lifestyle survey each household completed once. The household above is what that data looks like up close. It's also the first of seven case studies built the same way, one real dataset and one real business question each, so the habit you build in this chapter (reason through the ask, prototype in the notebook, promote what works into a tested module) is the one the other six will follow.

## 1. Problem framing

Here's the request, exactly as it would actually arrive. You're the data scientist on a retention team at a company sitting on data like this, purchase histories linked to a demographic survey, for a panel of customers. Your manager stops by your desk and says:

"We want to know which customers are worth investing more in."

That's the whole brief. No target variable, no metric, no mention of a model. This is what a real ask sounds like, and your job right now is to figure out what it actually means, not to open a notebook.

### What does "worth investing in" even mean?

Before you touch a single file, you need to pull that sentence apart. It's collapsing at least four different problems into one, and they are not interchangeable:

- Customers who already spend the most. A ranking of the past, nothing predictive about it.
- Customers likely to spend more going forward. A forecast.
- Customers likely to leave if nobody intervenes. A churn problem.
- Customers who look like existing high spenders but haven't been identified yet. A segmentation problem.

Each one is a legitimate data science problem. Each one produces a different target variable, a different model, and a different action the business takes on your output. Guess wrong here and you could spend six months building the right system for the wrong question.

You go back and ask a follow-up question. It turns out "worth investing in" means deciding who gets priority for a loyalty program with a fixed, limited budget. The team wants customers ranked by how much they're likely to spend going forward, using the demographic and behavioural data already on file, before that spending happens. Not a scoreboard of who has already spent the most.

That answer resolves the first fork: this is a forecast of future value, not a description of past value, which rules out simply sorting by historical total. It also starts pinning down [Chapter 30's five-component spec](../04-ml-intro/30-ml-workflow.qmd#translating-a-business-problem-into-a-machine-learning-problem), one piece at a time, following wherever your own follow-up questions lead next:

<div class='ark-activity'>
<span class='ark-activity-title'><i class="bi bi-puzzle-fill"></i> Activity: write the spec card before reading further</span><br><br>
Using [Chapter 30's five-component spec](../04-ml-intro/30-ml-workflow.qmd#translating-a-business-problem-into-a-machine-learning-problem) (target variable, task type and paradigm, success metric, acceptable threshold, data specification), write your own answer for this problem before reading the discussion below. Compare where you landed once you get there. If your target variable differs, that's fine, it's the kind of decision [Chapter 30's Pro Tip](../04-ml-intro/30-ml-workflow.qmd#translating-a-business-problem-into-a-machine-learning-problem) already warned about: agree it with the stakeholder who will actually use the model, because a model optimised toward the wrong objective will hit that objective and still miss the point.
</div>

**Target variable.** Not last month's spend, what you're predicting is next year's. You still need a window, a month, a year, a lifetime span all answer the question differently, and the right one depends on how often the loyalty program actually reallocates its budget. If the business revisits the list annually, that settles it: a household's spend over the coming twelve months.

**Task type and paradigm.** You're predicting a number from labelled historical examples where the outcome is already known. That's regression, supervised.

**Success metric.** The team isn't going to read predictions one row at a time, they're going to act on a ranked list. So the metric that matters is whether your model orders households correctly relative to each other, and whether the top decile it flags genuinely outspends everyone else. Mean absolute error is still worth tracking, mostly as a sanity check that the numbers themselves aren't nonsense.

**Acceptable threshold.** A model that can't beat "assume everyone spends the average" isn't worth shipping. Here's the real bar: households you rank in the top decile need to outspend a randomly chosen household by enough that the loyalty program's cost per member actually pays for itself, a number the business has to hand you, you can't derive it from the data alone.

**Data specification.** One demographic and lifestyle survey per household, answered once, and multi-year purchase history at the individual-transaction level, for 5,027 households.

::: {.callout-note collapse="true" icon=false}
## The resolved spec card

| Component | Specification |
|---|---|
| Target variable | Household's total spend over the coming 12 months |
| Task type and paradigm | Regression, supervised |
| Success metric | Rank quality (does the top decile of predicted spend actually outspend the rest), MAE as a secondary check |
| Acceptable threshold | Top-decile households must outspend a randomly chosen household by a margin the loyalty program's budget can justify |
| Data specification | One-time demographic and lifestyle survey plus multi-year, transaction-level purchase history, 5,027 U.S. households |
:::

That's the spec, and it took five follow-up questions to get there, not a target variable declared in the first paragraph. Every one of those questions is something a real stakeholder conversation would actually raise, and you'll ask versions of them again in every case study that follows.

Next: turning this into something durable enough that the rest of the project can't quietly drift from it, a Pydantic object, not a paragraph you could reinterpret three sections from now.

### Making the decision durable

Right now the spec card is a table in a notebook. Nothing stops the next person, or you, in three weeks, from reading "predicted spend" and quietly building a model for last month's spend instead, because nothing enforces which one was actually decided. A Pydantic object does. Once `settings.spec.target_variable` is a real attribute, every later section imports it instead of re-reading a paragraph and hoping it remembers correctly.

This is also where the practical decisions live, the ones that don't belong in a business spec but still need a fixed answer somewhere: which Harvard Dataverse files this pulls from, where they get cached, and which survey questions never make it into a feature, no matter how predictive they turn out to be.

In [ ]:
from pathlib import Path

from pydantic import BaseModel
from pydantic_settings import BaseSettings, SettingsConfigDict


class SpecCard(BaseModel):
    """The five-component spec resolved above, kept as data, not prose."""

    target_variable: str = "Household's total spend over the coming 12 months"
    task_type: str = "Regression, supervised"
    success_metric: str = "Rank quality (top-decile lift over random), MAE as a secondary check"
    acceptable_threshold: str = (
        "Top-decile households must outspend a random household by a margin the loyalty program's budget can justify"
    )
    data_specification: str = (
        "One-time demographic and lifestyle survey plus multi-year, "
        "transaction-level purchase history, 5,027 U.S. households"
    )


class SpendRegressionSettings(BaseSettings):
    """Paths, source data, and policy decisions for this case study."""

    model_config = SettingsConfigDict(env_file=".env", env_prefix="SPEND_", case_sensitive=False)

    spec: SpecCard = SpecCard()

    data_dir: Path = Path("data")
    dataverse_base_url: str = "https://dataverse.harvard.edu/api/access/datafile"
    purchases_file_id: int = 7616235  # amazon-purchases.csv, ~298.5 MB
    survey_file_id: int = 7616231  # survey.csv, tab-separated despite the .csv extension

    random_seed: int = 42

    excluded_survey_columns: list[str] = [
        "Q-substance-use-cigarettes",
        "Q-substance-use-marijuana",
        "Q-substance-use-alcohol",
        "Q-personal-diabetes",
        "Q-personal-wheelchair",
        "Q-sexual-orientation",
    ]


settings = SpendRegressionSettings()
settings.spec.target_variable

In [ ]:
from great_tables import GT, md as gt_md
import pandas as pd

from ark.plot.gt_style import themed_gt

LABELS = {
    "target_variable": "Target variable",
    "task_type": "Task type and paradigm",
    "success_metric": "Success metric",
    "acceptable_threshold": "Acceptable threshold",
    "data_specification": "Data specification",
}
spec_df = pd.DataFrame([{"Component": LABELS[k], "Specification": v} for k, v in settings.spec.model_dump().items()])
themed_gt(GT(spec_df), n_rows=len(spec_df)).tab_header(
    title=gt_md("**The spec card, as code**"),
    subtitle="Generated from settings.spec, not retyped from the table above",
)

Same five rows as the resolved spec card earlier in this chapter. The difference is where they came from: that one was a markdown table anyone could edit without noticing, this one is read straight off the object the rest of the pipeline will actually import.

This is what "promoted" means in practice. The two classes above are not staying in this notebook, they live at `ark/case_studies/spend_regression/config.py`, word for word. Proving that means checking the values survive the move, then using the real, imported class for everything from here on, not the copy defined above:

In [ ]:
before_spec = settings.spec.model_dump()
before_exclusions = settings.excluded_survey_columns

from ark.case_studies.spend_regression.config import SpendRegressionSettings

settings = SpendRegressionSettings()
settings.spec.model_dump() == before_spec and settings.excluded_survey_columns == before_exclusions

No test file for this one. Pydantic already validates the structure on construction, and there's no logic here yet worth a regression test, just decisions. That changes at the next step: downloading the actual data is where something can genuinely go wrong (a bad URL, a schema drift, a file Harvard renames), and where this chapter writes its first real test.

## 2. Data collection

The data specification says a one-time survey plus multi-year purchase history, for 5,027 households, hosted on Harvard Dataverse. Downloading a file isn't a data science problem, it's infrastructure, so you're not building it by hand in this notebook. It's a utility: written once, tested, imported everywhere it's needed, the same way you'd reach for a library function instead of re-deriving it.

In [ ]:
from ark.case_studies.spend_regression.data import load_survey

Behind that import is `download_dataverse_file`, which handles the parts that have nothing to do with this dataset specifically and everything to do with downloading files correctly:

- **Idempotent.** You won't wait through a 300 MB download twice, once a file is on disk, this skips the network entirely.
- **Streamed in chunks.** Not read into memory in one call, so a large file doesn't need to fit in RAM before it's even written to disk.
- **HTTPS-only, on purpose.** It refuses any URL that isn't `https://` before opening a connection. Dataverse would never hand back anything else, but you shouldn't write a function that fetches arbitrary URLs and quietly trusts whatever string it's given.
- **A plain header fixes a confusing error.** Dataverse's server rejects Python's default `urllib` user agent with a 403, even though the same URL works fine in `curl` or a browser. You send a normal-looking header instead, one line, and the "broken API" turns out to be a request that just looked wrong.

None of that is specific to spend prediction. You're looking at code worth writing once, testing, and never thinking about again. That's why it gets its own test, next.

This is also the repo's first real test file, `tests/case_studies/test_spend_regression.py`, three tests that never touch the network: one confirms the cache short-circuit works without a real download, one confirms the HTTPS guard rejects a bad scheme, one confirms the tab-separator handling on a small fixture file. Run it yourself here rather than take it on faith:

In [ ]:
!uv run pytest ../../tests/case_studies/test_spend_regression.py -v --override-ini=addopts=""

### Loading the survey

Before loading it: `survey.csv` is tab-separated. Nothing about the extension warns you, and `pd.read_csv` with default settings would silently parse it as one giant column instead of raising an error, a worse failure than a crash because it looks like it worked. `load_survey` already accounts for this.

In [ ]:
survey_df = load_survey(settings)
themed_gt(GT(survey_df.head(5)), n_rows=5).tab_header(
    title=gt_md("**First 5 survey responses**"),
    subtitle=f"{survey_df.shape[1]} columns per household",
)

Scroll through the columns above and you'll find `Q-demos-age`, `Q-demos-income`, `Q-demos-education` sitting a few columns away from `Q-substance-use-cigarettes`, `Q-personal-diabetes`, `Q-personal-wheelchair`, and `Q-sexual-orientation`. One survey, one consent form, questions about spending habits next to questions that have nothing to do with them. `excluded_survey_columns` in `config.py` is what keeps the second group out of the feature set, whether they're worth excluding on predictive grounds is a question for EDA, next.

## 3. EDA

[Chapter 16's Datasaurus Dozen](../01-python-basics/16-data-storytelling.ipynb#the-datasaurus-dozen-always-visualize) made this point once already: never trust a summary statistic you haven't plotted. You're about to load 1.85 million purchase records, the largest dataset this book has put in front of you, and the pull is to jump straight to a model. Look first.

<div class='ark-activity'>
<span class='ark-activity-title'><i class="bi bi-puzzle-fill"></i> Activity: decide what EDA has to answer before you run any of it</span><br><br>
Go back to <a href="../04-ml-intro/30-ml-workflow.qmd#translating-a-business-problem-into-a-machine-learning-problem">the resolved spec card</a>: target variable, task type, success metric, threshold, data spec. Before reading further, write down two or three things about this data you'd want to check before building a single feature, and *why* each one matters given that specific spec, not "what EDA usually covers." Compare your list against the four checks below.
</div>

::: {.callout-note collapse="true" icon=false}
## What this section actually checks, and why

This section checks four things, each one earns its place because of something in the spec card, not because it's customary EDA:

- **The shape of household spend.** The target itself. If it's skewed enough, a plain mean baseline misleads you and the choice of success metric stops being academic.
- **Whether spend changed by year.** The target is a forward-looking number. If a household's history looks different depending on when it was collected, that's a fact any feature you build has to respect.
- **Missingness in `Category` and `Title`.** Section 4 needs a category for every row. A blank one has to be handled on purpose, not discovered as a crash.
- **Whether the excluded survey columns actually cost you anything.** Section 2 left this open. It's answerable with the same data you're about to load.

If your list looked different, that's fine, the point isn't to predict this exact list, it's to arrive at EDA already knowing what question you're trying to answer instead of poking at charts until something looks interesting.
:::

In [ ]:
from lets_plot import (
    LetsPlot,
    aes,
    geom_bar,
    geom_line,
    geom_point,
    geom_vline,
    ggplot,
    labs,
    scale_x_continuous,
)

from ark.case_studies.spend_regression.data import load_purchases
from ark.plot.theme import modern_theme
from ark.plot.tokens import DANGER, INFO

LetsPlot.setup_html()

purchases_df = load_purchases(settings)
purchases_df.shape

Purchases are logged per line item, one row per thing bought. To see how spend behaves, you need one row per household first.

In [ ]:
hh_total = purchases_df.groupby("Survey ResponseID")["total"].sum().rename("total_spend").reset_index()
(
    ggplot(hh_total, aes(x="total_spend"))
    + geom_bar(stat="bin", fill=INFO, alpha=0.85, bins=60)
    + geom_vline(xintercept=16237.77, color=DANGER, linetype="dashed", size=1)
    + labs(
        title="Household lifetime spend",
        subtitle="Dashed line: the household from the opening of this chapter, 85th percentile",
        x="Total spend ($)",
        y="Households",
    )
    + modern_theme(grid=True)
)

Median household spend across the whole dataset is $5,706.99. The mean is $8,763.40, more than half again as much, which is what you'd expect from a right-skewed distribution: most households cluster well below the mean, and a smaller group of very high spenders pulls it upward. You can see where the household from the opening lands: the 85th percentile of lifetime spend, comfortably above average, but nowhere near the top. Whatever happened in November 2019 wasn't a top spender having a big month. It was an ordinary-to-good household having an extraordinary one.

This is also why Section 1's success metric was rank quality, not raw error. A skew this wide means a handful of households can swing a mean-based metric like MAE far more than they should, while barely moving where a household sits relative to the rest. You picked the metric before seeing this chart. The chart is why that choice holds up.

### The COVID-era shift

The household-level total hides when spending happened. Aggregate by year instead and you get a shape most of the developed world lived through directly.

In [ ]:
purchases_df["year"] = purchases_df["Order Date"].dt.year
yearly = (
    purchases_df[purchases_df["year"].between(2018, 2022)]
    .groupby(["Survey ResponseID", "year"])["total"]
    .sum()
    .reset_index()
    .groupby("year")["total"]
    .mean()
    .reset_index()
)
(
    ggplot(yearly, aes(x="year", y="total"))
    + geom_line(color=INFO, size=1.2)
    + geom_point(color=INFO, size=3)
    + scale_x_continuous(breaks=[2018, 2019, 2020, 2021, 2022], format="d")
    + labs(
        title="Mean annual household spend, 2018-2022",
        subtitle="2023-2024 excluded: data collection tapers off, not household behaviour",
        x="Year",
        y="Mean spend ($)",
    )
    + modern_theme(grid=True)
)

Mean annual spend rises every year from 2018 ($1,286) through 2021 ($2,287), nearly doubling, then holds roughly flat in 2022 ($2,204). None of this needed a model, a `groupby` and a chart found it. What it means for you as you build a spend-prediction model matters more: a household's 2018 history looks nothing like the same household's 2021 history, so the features you build in Section 4 need to account for when the data was collected, not only what was bought.

### Missingness

`Category` and `Title` are each missing on about 4.8% of rows, and it's almost always the same rows: 88,832 of the 89,458 records missing `Category` are also missing `Title`, over 99%. That's not data-entry noise scattered evenly across the file, you're looking at one class of purchase record, probably a discontinued listing or a third-party item Amazon never fully catalogued, that's incomplete in a consistent way.

In [ ]:
missing_df = pd.DataFrame(
    {
        "Column": ["Category", "Title"],
        "Missing (count)": [purchases_df["Category"].isna().sum(), purchases_df["Title"].isna().sum()],
        "Missing (%)": [
            round(purchases_df["Category"].isna().mean() * 100, 2),
            round(purchases_df["Title"].isna().mean() * 100, 2),
        ],
    }
)
themed_gt(GT(missing_df), n_rows=len(missing_df)).tab_header(
    title=gt_md("**Missing values**"),
    subtitle="Category and Title are missing on almost exactly the same rows",
)

You need a category for every row regardless of whether the source data has one. Section 4 handles that.

### Does any of this actually predict spend?

Section 2 left one question open for you: whether the columns `config.py` excludes, substance use, disability, sexual orientation, would have predicted spend better than the demographic columns that stayed in.

<div class='ark-activity'>
<span class='ark-activity-title'><i class="bi bi-puzzle-fill"></i> Activity: guess before you check</span><br><br>
Before running the next cell, write down a guess for each of the six excluded columns: do you expect smoking status, marijuana use, alcohol use, diabetes, wheelchair use, and sexual orientation to show a real gap in mean spend, or none worth mentioning? You have one piece of evidence already, the spec card's data specification says nothing about health or identity driving spend, which is exactly the kind of assumption worth checking rather than trusting.
</div>

In [ ]:
hh_spend = purchases_df.groupby("Survey ResponseID")["total"].sum().rename("total_spend")
survey_with_spend = survey_df.merge(hh_spend, on="Survey ResponseID", how="inner")

comparisons = [
    ("Q-substance-use-cigarettes", "No", "Yes"),
    ("Q-substance-use-marijuana", "No", "Yes"),
    ("Q-substance-use-alcohol", "No", "Yes"),
    ("Q-personal-diabetes", "No", "Yes"),
    ("Q-personal-wheelchair", "No", "Yes"),
    ("Q-sexual-orientation", "heterosexual (straight)", "LGBTQ+"),
]
rows = []
for col, baseline, comparison in comparisons:
    baseline_mean = survey_with_spend.loc[survey_with_spend[col] == baseline, "total_spend"].mean()
    comparison_mean = survey_with_spend.loc[survey_with_spend[col] == comparison, "total_spend"].mean()
    rows.append(
        {
            "Column": col,
            "Baseline": f"{baseline} (${baseline_mean:,.2f})",
            "Comparison": f"{comparison} (${comparison_mean:,.2f})",
            "Gap": f"{(comparison_mean - baseline_mean) / baseline_mean:+.2%}",
        }
    )
excluded_check = pd.DataFrame(rows)
themed_gt(GT(excluded_check), n_rows=len(excluded_check)).tab_header(
    title=gt_md("**Mean household spend by excluded column**"),
    subtitle="Checking whether the exclusion decision costs any real predictive power",
)

If you guessed "no meaningful gap anywhere," you were half right, and the half where you were wrong is the more important half. Three of the six show almost no gap: alcohol use, diabetes, and using a wheelchair. Two show a real one: households where someone smokes cigarettes or uses marijuana spend noticeably less, not more, and LGBTQ+ households spend about 17% less than heterosexual ones. You're not looking at rounding errors.

Which makes your exclusion decision more interesting, not less defensible. A real correlation between sexual orientation and spend doesn't mean sexual orientation causes spend, it almost certainly reflects something else these groups differ on that already has its own column, income, age, region, and a demographic proxy can pick up someone else's signal without you gaining anything a more direct feature wouldn't explain better. Even if it were a genuinely independent predictor, using a protected characteristic to price or prioritise a customer carries consequences beyond model accuracy, and [Chapter 30's spec card](../04-ml-intro/30-ml-workflow.qmd#translating-a-business-problem-into-a-machine-learning-problem) never asked what's technically permitted to use. It asked what the acceptable threshold and success metric are, not how to hit them by any means available. `excluded_survey_columns` was the right call before you had this table. Now you have evidence for it, not just a policy.

Section 4 turns all of this, the skew, the year effect, the missingness, into features a model can actually use.